# Gold Layer — Revenue Predictor

End-to-end ML pipeline: feature engineering → model training → evaluation → registry deployment.

**Goal:** Predict monthly revenue by product category using historical patterns, lagged revenue, order metrics, and seasonal features.

**Models:** XGBRegressor vs LinearRegression

**Prerequisites:** Phases 1-3 complete (setup, data creation, processing).

## 1. Setup

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window

from snowflake.ml.modeling.preprocessing import OrdinalEncoder, MinMaxScaler
from snowflake.ml.modeling.xgboost import XGBRegressor
from snowflake.ml.modeling.linear_model import LinearRegression
from snowflake.ml.modeling.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from snowflake.ml.registry import Registry
import math

session = get_active_session()
session.use_role("DEMO_ML_ENGINEER")
session.use_warehouse("DEMO_ML_WH")
session.use_database("MSFT_SNOWFLAKE_DEMO")
session.use_schema("ML")

print(session.sql("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()").collect())

## 2. Feature Engineering

In [ ]:
# Load source tables
orders_df = session.table("BRONZE.ORDERS")
order_items_df = session.table("BRONZE.ORDER_ITEMS")
products_df = session.table("BRONZE.PRODUCTS")

# Monthly aggregates by category
monthly_agg = (
    orders_df
    .join(order_items_df, on="ORDER_ID", how="inner")
    .join(products_df.select("PRODUCT_ID", "CATEGORY"), on="PRODUCT_ID", how="left")
    .with_column("ORDER_MONTH", F.date_trunc("month", F.col("ORDER_DATE")))
    .group_by("ORDER_MONTH", "CATEGORY")
    .agg(
        F.sum("LINE_TOTAL").alias("MONTHLY_REVENUE"),
        F.count_distinct("ORDERS.ORDER_ID").alias("ORDER_COUNT"),
        F.count_distinct("CUSTOMER_ID").alias("UNIQUE_CUSTOMERS"),
        F.sum("QUANTITY").alias("TOTAL_UNITS"),
        F.avg("DISCOUNT_PCT").alias("AVG_DISCOUNT"),
    )
)

# Lag and window features
month_window = Window.partition_by("CATEGORY").order_by("ORDER_MONTH")

features_raw = (
    monthly_agg
    .with_column("MONTH_NUM", F.month(F.col("ORDER_MONTH")))
    .with_column("QUARTER", F.quarter(F.col("ORDER_MONTH")))
    .with_column("YEAR", F.year(F.col("ORDER_MONTH")))
    .with_column("REVENUE_LAG_1", F.lag("MONTHLY_REVENUE", 1).over(month_window))
    .with_column("REVENUE_LAG_2", F.lag("MONTHLY_REVENUE", 2).over(month_window))
    .with_column("REVENUE_LAG_3", F.lag("MONTHLY_REVENUE", 3).over(month_window))
    .with_column("REVENUE_MA_3",
        F.avg("MONTHLY_REVENUE").over(month_window.rows_between(-3, -1)))
    .with_column("ORDER_COUNT_LAG_1", F.lag("ORDER_COUNT", 1).over(month_window))
    .with_column("REVENUE_GROWTH_PCT",
        F.when(F.col("REVENUE_LAG_1").is_not_null() & (F.col("REVENUE_LAG_1") > 0),
            F.round(
                (F.col("MONTHLY_REVENUE") - F.col("REVENUE_LAG_1"))
                / F.col("REVENUE_LAG_1") * 100, 2
            ))
        .otherwise(None))
    .filter(F.col("REVENUE_LAG_3").is_not_null())
)

print(f"Raw features: {features_raw.count():,} rows")

In [ ]:
# Encode CATEGORY
cat_encoder = OrdinalEncoder(input_cols=["CATEGORY"], output_cols=["CATEGORY_ENCODED"])
features = cat_encoder.fit(features_raw).transform(features_raw)

# Scale numerics
reg_numeric = [
    "ORDER_COUNT", "UNIQUE_CUSTOMERS", "TOTAL_UNITS", "AVG_DISCOUNT",
    "REVENUE_LAG_1", "REVENUE_LAG_2", "REVENUE_LAG_3", "REVENUE_MA_3",
    "ORDER_COUNT_LAG_1",
]
reg_scaled = [f"{c}_SCALED" for c in reg_numeric]

scaler = MinMaxScaler(input_cols=reg_numeric, output_cols=reg_scaled)
features = scaler.fit(features).transform(features)

# Final feature table
final_cols = (
    ["ORDER_MONTH", "CATEGORY", "MONTHLY_REVENUE"]
    + ["CATEGORY_ENCODED", "MONTH_NUM", "QUARTER", "YEAR"]
    + reg_scaled
    + ["REVENUE_GROWTH_PCT"]
)
features = features.select(final_cols)
features.write.save_as_table("ML.REVENUE_REGRESSION_FEATURES", mode="overwrite")

print(f"Saved ML.REVENUE_REGRESSION_FEATURES \u2014 {features.count():,} rows, {len(features.columns)} cols")

## 3. Explore Data

In [ ]:
features_df = session.table("ML.REVENUE_REGRESSION_FEATURES")
features_df.show(5)

FEATURE_COLS = [
    "CATEGORY_ENCODED", "MONTH_NUM", "QUARTER", "YEAR",
    "ORDER_COUNT_SCALED", "UNIQUE_CUSTOMERS_SCALED",
    "TOTAL_UNITS_SCALED", "AVG_DISCOUNT_SCALED",
    "REVENUE_LAG_1_SCALED", "REVENUE_LAG_2_SCALED",
    "REVENUE_LAG_3_SCALED", "REVENUE_MA_3_SCALED",
    "ORDER_COUNT_LAG_1_SCALED",
]
TARGET_COL = "MONTHLY_REVENUE"

print("\nTarget distribution:")
features_df.select(TARGET_COL).describe().show()

## 4. Train / Test Split

Chronological split — earlier months for training, last ~20% for testing.

In [ ]:
month_list = features_df.select("ORDER_MONTH").distinct().sort("ORDER_MONTH").collect()
cutoff_idx = int(len(month_list) * 0.8)
cutoff_date = month_list[cutoff_idx]["ORDER_MONTH"]
print(f"Cutoff date: {cutoff_date}")

train_df = features_df.filter(F.col("ORDER_MONTH") < cutoff_date)
test_df = features_df.filter(F.col("ORDER_MONTH") >= cutoff_date)

print(f"Training: {train_df.count():,} rows")
print(f"Test:     {test_df.count():,} rows")

print("\nTarget (MONTHLY_REVENUE) \u2014 Training:")
train_df.select(TARGET_COL).describe().show()

## 5. Model 1 \u2014 XGBoost Regressor

In [ ]:
xgb_model = XGBRegressor(
    input_cols=FEATURE_COLS,
    label_cols=[TARGET_COL],
    output_cols=["XGB_PREDICTED"],
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
)
xgb_model.fit(train_df)

xgb_preds = xgb_model.predict(test_df)

xgb_mae = mean_absolute_error(df=xgb_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["XGB_PREDICTED"])
xgb_mse = mean_squared_error(df=xgb_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["XGB_PREDICTED"])
xgb_rmse = math.sqrt(xgb_mse)
xgb_r2 = r2_score(df=xgb_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["XGB_PREDICTED"])

xgb_mape_df = (
    xgb_preds
    .with_column("APE", F.abs(F.col(TARGET_COL) - F.col("XGB_PREDICTED")) / F.col(TARGET_COL) * 100)
    .select(F.avg("APE").alias("MAPE"))
    .collect()
)
xgb_mape = xgb_mape_df[0]["MAPE"]

print(f"XGBRegressor Results:")
print(f"  MAE:  ${xgb_mae:,.2f}")
print(f"  RMSE: ${xgb_rmse:,.2f}")
print(f"  R\u00b2:   {xgb_r2:.4f}")
print(f"  MAPE: {xgb_mape:.2f}%")

## 6. Model 2 \u2014 Linear Regression (Baseline)

In [ ]:
lr_model = LinearRegression(
    input_cols=FEATURE_COLS,
    label_cols=[TARGET_COL],
    output_cols=["LR_PREDICTED"],
)
lr_model.fit(train_df)

lr_preds = lr_model.predict(test_df)

lr_mae = mean_absolute_error(df=lr_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["LR_PREDICTED"])
lr_mse = mean_squared_error(df=lr_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["LR_PREDICTED"])
lr_rmse = math.sqrt(lr_mse)
lr_r2 = r2_score(df=lr_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["LR_PREDICTED"])

lr_mape_df = (
    lr_preds
    .with_column("APE", F.abs(F.col(TARGET_COL) - F.col("LR_PREDICTED")) / F.col(TARGET_COL) * 100)
    .select(F.avg("APE").alias("MAPE"))
    .collect()
)
lr_mape = lr_mape_df[0]["MAPE"]

print(f"LinearRegression Results:")
print(f"  MAE:  ${lr_mae:,.2f}")
print(f"  RMSE: ${lr_rmse:,.2f}")
print(f"  R\u00b2:   {lr_r2:.4f}")
print(f"  MAPE: {lr_mape:.2f}%")

## 7. Compare Models

In [ ]:
print(f"{'Metric':<12} {'XGBRegressor':>16} {'LinearRegression':>18}")
print("-" * 48)
print(f"{'MAE':<12} ${xgb_mae:>14,.2f} ${lr_mae:>16,.2f}")
print(f"{'RMSE':<12} ${xgb_rmse:>14,.2f} ${lr_rmse:>16,.2f}")
print(f"{'R\u00b2':<12} {xgb_r2:>15.4f} {lr_r2:>17.4f}")
print(f"{'MAPE':<12} {xgb_mape:>14.2f}% {lr_mape:>16.2f}%")

best_name = "XGBRegressor" if xgb_r2 >= lr_r2 else "LinearRegression"
best_r2 = max(xgb_r2, lr_r2)
print(f"\n\u2192 Best model: {best_name} (R\u00b2={best_r2:.4f})")

## 8. Save Predictions & Feature Importance

In [ ]:
# Save XGB predictions with residuals
xgb_output = (
    xgb_preds
    .with_column("MODEL_NAME", F.lit("XGBRegressor"))
    .with_column("MODEL_VERSION", F.lit("v1"))
    .with_column("RESIDUAL", F.col(TARGET_COL) - F.col("XGB_PREDICTED"))
    .with_column("PCT_ERROR",
        F.round(F.abs(F.col(TARGET_COL) - F.col("XGB_PREDICTED")) / F.col(TARGET_COL) * 100, 2))
    .select("ORDER_MONTH", "CATEGORY", "MONTHLY_REVENUE",
            "XGB_PREDICTED", "RESIDUAL", "PCT_ERROR",
            "MODEL_NAME", "MODEL_VERSION")
    .with_column_renamed("XGB_PREDICTED", "PREDICTED_REVENUE")
)
xgb_output.write.save_as_table("ML.REGRESSION_PREDICTIONS", mode="overwrite")
print(f"Saved ML.REGRESSION_PREDICTIONS \u2014 {xgb_output.count():,} rows")

print("\nSample Predictions vs Actuals:")
session.table("ML.REGRESSION_PREDICTIONS").sort("ORDER_MONTH", "CATEGORY").show(10)

# Feature importance
try:
    sklearn_model = xgb_model.to_sklearn()
    importances = sklearn_model.feature_importances_
    print("\nFeature Importance (XGBRegressor):")
    for feat, imp in sorted(zip(FEATURE_COLS, importances), key=lambda x: x[1], reverse=True):
        bar = "\u2588" * int(imp * 50)
        print(f"  {feat:<35} {imp:.4f} {bar}")
except Exception as e:
    print(f"Could not extract feature importance: {e}")

## 9. Residual Analysis

In [ ]:
residuals = session.table("ML.REGRESSION_PREDICTIONS")

residuals.select(
    F.avg("RESIDUAL").alias("MEAN_RESIDUAL"),
    F.stddev("RESIDUAL").alias("STD_RESIDUAL"),
    F.min("RESIDUAL").alias("MIN_RESIDUAL"),
    F.max("RESIDUAL").alias("MAX_RESIDUAL"),
    F.avg("PCT_ERROR").alias("AVG_PCT_ERROR"),
).show()

print("Residuals by Category:")
residuals.group_by("CATEGORY").agg(
    F.avg("PCT_ERROR").alias("AVG_PCT_ERROR"),
    F.avg("RESIDUAL").alias("AVG_RESIDUAL"),
    F.count("*").alias("COUNT"),
).sort("CATEGORY").show()

## 10. Register Model

In [ ]:
registry = Registry(session=session, database_name="MSFT_SNOWFLAKE_DEMO", schema_name="ML")

model_version = registry.log_model(
    model=xgb_model,
    model_name="REVENUE_PREDICTOR",
    version_name="v1",
    comment="XGBRegressor for monthly revenue prediction by product category. "
            "Uses lagged revenue, order metrics, and seasonal features.",
    metrics={
        "mae": xgb_mae,
        "rmse": xgb_rmse,
        "r2": xgb_r2,
        "mape": xgb_mape,
        "training_rows": train_df.count(),
        "test_rows": test_df.count(),
        "n_features": len(FEATURE_COLS),
        "algorithm": "XGBRegressor",
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
    },
    sample_input_data=train_df.select(FEATURE_COLS).limit(10),
)

# Set as default
registry.get_model("REVENUE_PREDICTOR").default = "v1"

print(f"Registered: REVENUE_PREDICTOR v1")
print(f"Metrics: {model_version.show_metrics()}")

## 11. Inference from Registry

In [ ]:
loaded_model = registry.get_model("REVENUE_PREDICTOR").version("v1")

sample = test_df.select(FEATURE_COLS).limit(5)
results = loaded_model.run(sample, function_name="predict")
results.show()

## SQL Inference

After registration, call the model directly from SQL:

```sql
SELECT
    r.ORDER_MONTH,
    r.CATEGORY,
    r.MONTHLY_REVENUE AS ACTUAL_REVENUE,
    ML.REVENUE_PREDICTOR!PREDICT(
        r.CATEGORY_ENCODED, r.MONTH_NUM, r.QUARTER, r.YEAR,
        r.ORDER_COUNT_SCALED, r.UNIQUE_CUSTOMERS_SCALED,
        r.TOTAL_UNITS_SCALED, r.AVG_DISCOUNT_SCALED,
        r.REVENUE_LAG_1_SCALED, r.REVENUE_LAG_2_SCALED,
        r.REVENUE_LAG_3_SCALED, r.REVENUE_MA_3_SCALED,
        r.ORDER_COUNT_LAG_1_SCALED
    ) AS PREDICTED_REVENUE
FROM ML.REVENUE_REGRESSION_FEATURES r
ORDER BY r.ORDER_MONTH DESC
LIMIT 10;
```